In [2]:
# after vector similairty search
reduce_length_1 = 8000 # chars

# after reranker
condense_length_2 = 4000 # chars

In [3]:
from dotenv import load_dotenv
import os
import sys

load_dotenv()
# set the python path
sys.path.append(os.getenv('PYTHON_PATH'))

In [4]:
from s3 import download_model_list_from_s3
from langchain.docstore.document import Document
from format import format_topic
from langchain_openai import OpenAIEmbeddings
from portkey_ai import PORTKEY_GATEWAY_URL, createHeaders, AsyncPortkey
import numpy as np
import pandas as pd
import cohere  
from portkey_ai import Portkey

In [5]:
def to_single_string(documents):
        if type (documents[0]) == dict:
            return "\n\n".join([doc['page_content'] for doc in documents])
        else:
            return "\n\n".join([
                doc.page_content 
                for doc in documents
            ]).strip()
        
def save_markdown_file(topic_docs, filename):    
    markdown = format_topic(topic_docs)
    # make folder if it doesnt exist
    with open(filename, 'w') as f:
        f.write(markdown)
        
def reduce_docs_to_length(docs, key, length, remove_highest_first=False):
    if type(docs[0]) == dict:
        doc_dicts = docs
    else:
        doc_dicts = [doc.to_dict() for doc in docs]

    def get_nested_value(doc_dict, key_path):
        value = doc_dict
        for key in key_path:
            if isinstance(value, dict):
                value = value.get(key)
            else:
                return None
        return value

    doc_dict_pairs = [(i, doc_dict, get_nested_value(doc_dict, key)) for i, doc_dict in enumerate(doc_dicts)]
    
    # Filter out documents with None as the key value
    doc_dict_pairs = [pair for pair in doc_dict_pairs if pair[2] is not None]
    
    # Sort the document-key pairs based on the key value
    if not remove_highest_first:
        doc_dict_pairs.sort(key=lambda x: x[2], reverse=True)
    else:
        doc_dict_pairs.sort(key=lambda x: x[2])

    reduced_docs = []
    current_length = 0

    for i, doc_dict, _ in doc_dict_pairs:
        doc_length = len(doc_dict.get('page_content', ''))
        if current_length + doc_length <= length:
            reduced_docs.append(doc_dict)
            current_length += doc_length

    # Sort the reduced_docs list based on the original order of docs
    reduced_docs.sort(key=lambda doc: docs.index(doc))
    
    # # warning if docs not reduced to length
    # if len(to_single_string(reduced_docs)) > length:
    #     print(f"Warning: Documents not reduced to length. Length: {len(to_single_string(reduced_docs))} > {length}")

    return reduced_docs


In [6]:
# test data
topics = [
    {
        "presentation_id": "018efa53-2169-7dc0-ad63-e7084a09afc4",
        "presentation_topic": "Exploring Cultural Contrasts: Japan vs. the USA - Insights for American Expats",
        "presentation_audience": "American expats living in Japan",
        "presentation_goal": "To inform and offer practical tips to American expats living in Japan about the cultural contrasts between Japan and the USA",
        "id": "066236b0-e1e0-77d4-8000-590e777d7d29",
        "name": "Cultural Influence on Help Seeking and Support from Professionals and Others"
    },
    {
        "presentation_id": "018efa53-2169-7dc0-ad63-e7084a09afc4",
        "presentation_topic": "Exploring Cultural Contrasts: Japan vs. the USA - Insights for American Expats",
        "presentation_audience": "American expats living in Japan",
        "presentation_goal": "To inform and offer practical tips to American expats living in Japan about the cultural contrasts between Japan and the USA",
        "id": "066236b0-e1e0-7967-8000-39e03b240974",
        "name": "Indirect Communication in Japanese Culture"
    },
    {
        "presentation_id": "018efa6a-2208-79d3-a905-958f91b06da7",
        "presentation_topic": "Antitrust pressure against Apple: History, Antitrust Arguments, and Apple's Counterarguments",
        "presentation_audience": "Tech enthusiasts and consumers",
        "presentation_goal": "Just inform the audience without a persuasive bias",
        "id": "06623708-33af-7e71-8000-883e8038cf64",
        "name": "Apple's Antitrust Allegations and Competition Concerns"
    },
    {
        "presentation_id": "018efa76-2369-7861-b1b7-592b3e5c275f",
        "presentation_topic": "The 7 Natural Wonders of the World",
        "presentation_audience": "Travelers and nature enthusiasts",
        "presentation_goal": "To inform the audience about the 7 natural wonders of the world, their locations, and what makes them special",
        "id": "0662373e-4a90-7df1-8000-eaae8e2f7505",
        "name": "Mount Everest - The Tallest Peak in the World"
    },
    {
        "presentation_id": "018efa76-2369-7861-b1b7-592b3e5c275f",
        "presentation_topic": "The 7 Natural Wonders of the World",
        "presentation_audience": "Travelers and nature enthusiasts",
        "presentation_goal": "To inform the audience about the 7 natural wonders of the world, their locations, and what makes them special",
        "id": "06623734-5459-7ea5-8000-1842ed1b986e",
        "name": "Youngest Volcano in Northern Hemisphere - Paricutin, Mexico"
    },
    {
        "presentation_id": "018efa80-4216-71c9-b37c-b55cb8e17a2d",
        "presentation_topic": "Understanding the Transformer Model Architecture: Revolutionizing Natural Language Processing\n\nTo give an informative high-level overview of the Transformer model architecture.",
        "presentation_audience": "Artificial Intelligence and Machine Learning researchers and students",
        "presentation_goal": "To give an informative high-level overview of the Transformer model architecture.",
        "id": "0662375b-e9fe-7154-8000-0e67ffd84570",
        "name": "Multi-Head Attention Mechanisms in Transformer Architecture"
    }
]

In [7]:
# topic = topics[5]
topic = topics[0]

In [8]:
# Get chunked and labeled documents from S3 for whole presentation
all_docs = await download_model_list_from_s3(Document, f"{topic['presentation_id']}/all_sources_chunked_docs_with_topics.json")

#Get topic docs
topic_docs = [doc for doc in all_docs if 'topic' in doc.metadata and doc.metadata['topic']['name'] == topic['name']]
topic_docs = [dict(doc) for doc in topic_docs]

print(f"Original character length of topic docs: {len(to_single_string(topic_docs))}")
print(f"Original number of topic docs: {len(topic_docs)}")
print(f"Original number of sources: {len(set(doc['metadata']['source']['id'] for doc in topic_docs))}")

Original character length of topic docs: 23628
Original number of topic docs: 61
Original number of sources: 1


In [9]:
folder = topic['name'].replace(' ', '_')
if not os.path.exists(folder):
        os.makedirs(folder)
save_markdown_file(topic_docs, f"{folder}/original.md")

In [10]:

# reduce by probability
reduced_by_probability_docs = reduce_docs_to_length(topic_docs.copy(), ['metadata','topic', 'probability'], reduce_length_1, remove_highest_first=False)
print(f"Character length of topic docs after reduction by probability: {len(to_single_string(reduced_by_probability_docs))}")
print(f"Number of topic docs after reduction by probability: {len(reduced_by_probability_docs)}")
print(f"Number of sources after reduction by probability: {len(set(doc['metadata']['source']['id'] for doc in reduced_by_probability_docs))}")
print(f"Minimum probability increased from {min(doc['metadata']['topic']['probability'] for doc in topic_docs)} to {min(doc['metadata']['topic']['probability'] for doc in reduced_by_probability_docs)}")
folder = topic['name'].replace(' ', '_')
if not os.path.exists(folder):
        os.makedirs(folder)
save_markdown_file(reduced_by_probability_docs, f"{folder}/probability_reduced.md")

Character length of topic docs after reduction by probability: 7897
Number of topic docs after reduction by probability: 21
Number of sources after reduction by probability: 1
Minimum probability increased from 0.526952128058686 to 0.9632748540933228


In [11]:
# reduce by vector similarity 
# vectorize
portkeyHeaders = createHeaders(
    provider="openai",
    api_key=os.getenv("PORTKEY_API_KEY"),
)
openai_embeddings = OpenAIEmbeddings(model="text-embedding-3-small", dimensions=512, base_url=PORTKEY_GATEWAY_URL, default_headers=portkeyHeaders)
vectors = await openai_embeddings.aembed_documents([doc['page_content'] for doc in topic_docs])
doc_embeddings = np.array(vectors)

query_embedding = await openai_embeddings.aembed_query(topic['name'])

# try 3 different metrics
cosine_similarities = np.dot(doc_embeddings, query_embedding) / (np.linalg.norm(doc_embeddings, axis=1) * np.linalg.norm(query_embedding))
euclidean_distances = np.linalg.norm(doc_embeddings - query_embedding, axis=1)
dot_products = np.dot(doc_embeddings, query_embedding)

# add the metrics and the embeddings
assert len(topic_docs) == len(doc_embeddings) == len(cosine_similarities) == len(euclidean_distances) == len(dot_products)
for i in range(len(topic_docs)):
    doc = topic_docs[i]
    doc['embedding'] = doc_embeddings[i]
    doc['metadata']['topic']['cosine_similarity'] = cosine_similarities[i]
    doc['metadata']['topic']['euclidean_distance'] = euclidean_distances[i]
    doc['metadata']['topic']['dot_product'] = dot_products[i]

In [12]:
reduced_by_cosine_similarity_docs = reduce_docs_to_length(topic_docs.copy(), ['metadata','topic', 'cosine_similarity'], reduce_length_1, remove_highest_first=False)
print(f"Character length of topic docs after reduction by cosine similarity: {len(to_single_string(reduced_by_cosine_similarity_docs))}")
print(f"Number of topic docs after reduction by cosine similarity: {len(reduced_by_cosine_similarity_docs)}")
print(f"Number of sources after reduction by cosine similarity: {len(set(doc['metadata']['source']['id'] for doc in reduced_by_cosine_similarity_docs))}")
print(f"Minimum cosine similarity increased from {min(doc['metadata']['topic']['cosine_similarity'] for doc in topic_docs)} to {min(doc['metadata']['topic']['cosine_similarity'] for doc in reduced_by_cosine_similarity_docs)}")

folder = topic['name'].replace(' ', '_')
if not os.path.exists(folder):
        os.makedirs(folder)
save_markdown_file(reduced_by_cosine_similarity_docs, f"{folder}/cosine_reduced.md")

Character length of topic docs after reduction by cosine similarity: 7973
Number of topic docs after reduction by cosine similarity: 21
Number of sources after reduction by cosine similarity: 1
Minimum cosine similarity increased from 0.46766132525847576 to 0.6761891022548556


In [13]:
reduced_by_euclidean_distance_docs = reduce_docs_to_length(topic_docs.copy(), ['metadata','topic', 'euclidean_distance'], reduce_length_1, remove_highest_first=True)
print(f"Character length of topic docs after reduction by euclidean distance: {len(to_single_string(reduced_by_euclidean_distance_docs))}")
print(f"Number of topic docs after reduction by euclidean distance: {len(reduced_by_euclidean_distance_docs)}")
print(f"Number of sources after reduction by euclidean distance: {len(set(doc['metadata']['source']['id'] for doc in reduced_by_euclidean_distance_docs))}")
print(f"Maximum euclidean distance decreased from {max(doc['metadata']['topic']['euclidean_distance'] for doc in topic_docs)} to {max(doc['metadata']['topic']['euclidean_distance'] for doc in reduced_by_euclidean_distance_docs)}")

folder = topic['name'].replace(' ', '_')
if not os.path.exists(folder):
        os.makedirs(folder)
save_markdown_file(reduced_by_euclidean_distance_docs, f"{folder}/euclidean_reduced.md")

Character length of topic docs after reduction by euclidean distance: 7973
Number of topic docs after reduction by euclidean distance: 21
Number of sources after reduction by euclidean distance: 1
Maximum euclidean distance decreased from 1.0318320354995043 to 0.8047495234483146


In [14]:
reduced_by_dot_product_docs = reduce_docs_to_length(topic_docs.copy(), ['metadata','topic', 'dot_product'], reduce_length_1, remove_highest_first=False)
print(f"Character length of topic docs after reduction by dot product: {len(to_single_string(reduced_by_dot_product_docs))}")
print(f"Number of topic docs after reduction by dot product: {len(reduced_by_dot_product_docs)}")
print(f"Number of sources after reduction by dot product: {len(set(doc['metadata']['source']['id'] for doc in reduced_by_dot_product_docs))}")
print(f"Minimum dot product increased from {min(doc['metadata']['topic']['dot_product'] for doc in topic_docs)} to {min(doc['metadata']['topic']['dot_product'] for doc in reduced_by_dot_product_docs)}")

folder = topic['name'].replace(' ', '_')
if not os.path.exists(folder):
        os.makedirs(folder)
save_markdown_file(reduced_by_dot_product_docs, f"{folder}/dot_reduced.md")


Character length of topic docs after reduction by dot product: 7973
Number of topic docs after reduction by dot product: 21
Number of sources after reduction by dot product: 1
Minimum dot product increased from 0.46766132525847626 to 0.6761891022548565


In [15]:
topic_docs[0]

{'page_content': 'Previous research has found cultural differences in the frequency of support seeking. Asians and Asian Americans report seeking support from their close others to deal with their stress less often compared to European Americans. Similarly, other research on professional help seeking has shown that Asians and Asian Americans are less likely than European Americans to seek professional psychological help.',
 'metadata': {'headers': {},
  'source': {'title': 'tmpzur1rz40',
   'url': 'file:///var/folders/jp/99fbbz9959d892jf52cv2c3h0000gn/T/tmpzur1rz40',
   'type': '/var/folders/jp/99fbbz9959d892jf52cv2c3h0000gn/T/tmpzur1rz40',
   'extraction_method': 'openparse',
   'id': '018efa58-8a02-7e98-8bc3-3118629ad50e'},
  'topic': {'number': 0,
   'name': 'Cultural Influence on Help Seeking and Support from Professionals and Others',
   'probability': 0.8706837375695593,
   'cosine_similarity': 0.6431448844803225,
   'euclidean_distance': 0.8448137256456931,
   'dot_product': 0.6

In [17]:
# Reduce further by eucledean distance
reduced_further_by_euclidean_distance_docs = reduce_docs_to_length(reduced_by_euclidean_distance_docs.copy(), ['metadata','topic', 'euclidean_distance'], condense_length_2, remove_highest_first=True)
print(f"Character length of topic docs after reduction by euclidean distance: {len(to_single_string(reduced_further_by_euclidean_distance_docs))}")
print(f"Number of topic docs after reduction by euclidean distance: {len(reduced_further_by_euclidean_distance_docs)}")
print(f"Number of sources after reduction by euclidean distance: {len(set(doc['metadata']['source']['id'] for doc in reduced_further_by_euclidean_distance_docs))}")
print(f"Maximum euclidean distance decreased from {max(doc['metadata']['topic']['euclidean_distance'] for doc in reduced_by_euclidean_distance_docs)} to {max(doc['metadata']['topic']['euclidean_distance'] for doc in reduced_further_by_euclidean_distance_docs)}")

folder = topic['name'].replace(' ', '_')
if not os.path.exists(folder):
        os.makedirs(folder)
save_markdown_file(reduced_further_by_euclidean_distance_docs, f"{folder}/euclidean_reduced_2.md")

Character length of topic docs after reduction by euclidean distance: 4004
Number of topic docs after reduction by euclidean distance: 11
Number of sources after reduction by euclidean distance: 1
Maximum euclidean distance decreased from 0.8047495234483146 to 0.7755017071479506


In [29]:
# Reduce further by reranker


In [18]:
# Show the percentage of the documents that are over 512 cohere tokens
co = cohere.Client()

# count query tokens
response = co.tokenize(text=f"What can be said about {topic['name']}?", model="rerank-english-v2.0", offline=False)
query_tokens = len(response.tokens)

for doc in reduced_by_euclidean_distance_docs:
    response = co.tokenize(text=doc['page_content'], model="rerank-english-v2.0", offline=False)
    if len(response.tokens) + query_tokens > 512:
        print(f"Document {doc.metadata['source']['title']} has {len(response.tokens)} tokens")

In [24]:
# Initialize the Portkey client
cohere_portkey = Portkey(
    api_key=os.getenv("PORTKEY_API_KEY"), 
    virtual_key="trial-cohere-679b13" 
)

completion = cohere_portkey.post(
    '/rerank',
    return_documents=False,
    max_chunks_per_doc=1,
    model="rerank-english-v2.0",
    query=f"What can be said about {topic['name']}?",
    documents=[doc['page_content'] for doc in reduced_by_euclidean_distance_docs]
)

for result in completion.results:
    reduced_by_euclidean_distance_docs[result["index"]]['metadata']['topic']["rerank_score"] = result["relevance_score"]
    
reduced_by_rerank_score_docs = reduce_docs_to_length(reduced_by_euclidean_distance_docs.copy(), ['metadata','topic', 'rerank_score'], condense_length_2, remove_highest_first=False)
print(f"Character length of topic docs after reduction by rerank score: {len(to_single_string(reduced_by_rerank_score_docs))}")
print(f"Number of topic docs after reduction by rerank score: {len(reduced_by_rerank_score_docs)}")
print(f"Number of sources after reduction by rerank score: {len(set(doc['metadata']['source']['id'] for doc in reduced_by_rerank_score_docs))}")
print(f"Minimum rerank score increased from {min(doc['metadata']['topic']['rerank_score'] for doc in reduced_by_euclidean_distance_docs)} to {min(doc['metadata']['topic']['rerank_score'] for doc in reduced_by_rerank_score_docs)}")

folder = topic['name'].replace(' ', '_')
if not os.path.exists(folder):
        os.makedirs(folder)
save_markdown_file(reduced_by_rerank_score_docs, f"{folder}/rerank_reduced_2.md")



Character length of topic docs after reduction by rerank score: 3957
Number of topic docs after reduction by rerank score: 12
Number of sources after reduction by rerank score: 1
Minimum rerank score increased from 0.97986126 to 0.99203116


In [22]:
reduced_by_euclidean_distance_docs[0]

{'page_content': 'The present research examined another explanation for cultural differences in professional help seeking. We predicted that the observed cultural difference in professional help seek- ing is an extension of culture-specific interpersonal relationship patterns.',
 'metadata': {'headers': {},
  'source': {'title': 'tmpzur1rz40',
   'url': 'file:///var/folders/jp/99fbbz9959d892jf52cv2c3h0000gn/T/tmpzur1rz40',
   'type': '/var/folders/jp/99fbbz9959d892jf52cv2c3h0000gn/T/tmpzur1rz40',
   'extraction_method': 'openparse',
   'id': '018efa58-8a02-7e98-8bc3-3118629ad50e'},
  'topic': {'number': 0,
   'name': 'Cultural Influence on Help Seeking and Support from Professionals and Others',
   'probability': 0.7755798961178001,
   'cosine_similarity': 0.6992985511053074,
   'euclidean_distance': 0.7755017071479506,
   'dot_product': 0.699298551105308,
   'rerank_score': 0.9956006}},
 'type': 'Document',
 'embedding': array([ 3.22501088e-02, -1.59057261e-02,  4.79716007e-02,  6.597